<div style="display: flex; flex-direction: column; align-items: center; justify-content: center; text-align: center;">
    
<img src="./image/CESIPATH-removebg-preview.png" alt="Logo ADEME" width="200"/>
<br>    

# 🌿Livrable 1 : Modélisation formelle
# CESIPATH

### Optimisation des tournées de livraison pour la transition écologique
    
<br>

<img src="./image/ADEME-removebg-preview.png" alt="Logo ADEME" width="100"/>
<br>

**Appel à manifestation d'intérêt : Mobilité Multimodale Intelligente**

<br><br><br>

</div>

**Structure :** CesiCDP <br>
**Équipe Projet :** LUU Philippe, AFANE Youcef, RABATEL Antonin, RIVET Alexandre <br>
**Date :** Avril 2026

<br><br>

---


Ce document présente la modélisation mathématique d'un problème d'optimisation de tournées de livraison. Un véhicule unique, de capacité limitée, effectue plusieurs sous-tournées depuis un dépôt central. L'objectif est de minimiser le coût total des trajets en milieu urbain, en respectant des contraintes de restrictions sur les routes, de capacité de chargement et de pondération dynamique des coûts.

## Versions du document : 

Ce tableau retrace toutes les versions du document :

|Version|Date|Auteur(s)|Objet|
|----|----|----|----|
|1.0|02/04/2026|Equipe projet|Création du document|

# Projet ADEME — Optimisation des Tournées de Livraison
---

## 1. Contexte et enjeux

### 1.1 La mission ADEME

L'ADEME (Agence de l'Environnement et de la Maîtrise de l'Énergie) a lancé un appel à manifestation d'intérêt pour développer de nouvelles solutions de mobilité durable adaptées à différents types de territoires. L'enjeu dépasse la simple optimisation logistique : chaque kilomètre superflu parcouru par un véhicule de livraison représente une émission de CO₂ évitable, une consommation de carburant gaspillée et un temps de livraison accru.

Le rôle de notre équipe, CesiCDP, est de répondre à cet appel avec une solution algorithmique visant à minimiser les coûts opérationnels lors des tournées de livraison.

### 1.2 Un problème enrichi de contraintes réelles

Les tournées de livraison en conditions réelles ne se réduisent pas à un simple graphe abstrait. Trois contraintes opérationnelles majeures s'imposent dans le cadre de ce projet :

1. **Restrictions et surcoûts sur les arêtes** : certaines routes sont interdites (travaux, tonnage) ou plus coûteuses (péages, zones réglementées)
2. **Capacité de chargement du véhicule** : le véhicule ne peut transporter qu'une quantité limitée de marchandises
3. **Pondération dynamique des routes** : les coûts de trajet varient dans le temps (trafic, météo, accidents...)

Ces contraintes seront formalisées mathématiquement dans les sections suivantes.

## 2. Modélisation formelle

### 2.1 Représentation en graphe

Le réseau de livraison est modélisé par un graphe non orienté pondéré :

$$G = (V, E, w_0)$$

où :
- $V = \{v_0, v_1, \dots, v_{n-1}\}$ est l'ensemble des sommets, avec $|V| = n$
  - $v_0$ désigne le dépôt (point de départ et de retour du véhicule)
  - $v_1, \dots, v_{n-1}$ désignent les clients à livrer
- $E \subseteq \{\{v_i, v_j\} \mid i \neq j\}$ est l'ensemble des arêtes représentant les routes existantes
- $w_0 : E \rightarrow \mathbb{R}_+^*$ est la fonction de coût de base, associant à chaque arête un coût entier positif


**Complétion du graphe :** pour disposer d'un coût entre toute paire de sommets, on complète le graphe à partir des plus courts chemins calculés par l'algorithme de Dijkstra<sup>1</sup>. Pour chaque paire de sommets sans route directe exploitable, on remplace l'absence d'arête par le coût du plus court chemin dans $G$. On obtient ainsi un graphe complet :

$$G' = (V, E', w')$$

où $E' = \{\{v_i, v_j\} \mid i \neq j\}$ et $w'(\{v_i, v_j\})$ est le coût du plus court chemin entre $v_i$ et $v_j$ dans $G$.

La construction de $G'$ repose sur $n$ exécutions de Dijkstra, une par sommet source. Fredman et Tarjan<sup>2</sup> donnent une analyse classique de Dijkstra avec file de priorité, en $O((m+n)\log n)$. Dans notre implémentation matricielle, chaque sommet peut être relié à jusqu'à $n$ voisins, donc $m = O(n^2)$ ; on en déduit un coût en $O(n^2 \log n)$ par source, puis en $O(n^3 \log n)$ pour compléter tout le graphe. Cette transformation reste polynomiale.

---

### 2.2 Données associées aux clients

Tous les clients ont la même demande, déterminée par un tirage aléatoire unique :

$$\delta \in [1, 20] \cap \mathbb{R}_+$$

Chaque client $v_i \in V \setminus \{v_0\}$ a donc une demande $\delta_i = \delta$. Cette hypothèse de demande uniforme simplifie la contrainte de capacité : le nombre maximal de clients par sous-tournée est $ \frac{Q}{\delta}$.

---

### 2.3 Modélisation des contraintes

#### Contrainte 1 — Restrictions et surcoûts sur les arêtes

Chaque arête $\{v_i, v_j\} \in E$ est associée à un statut :

$$\text{statut}(\{v_i, v_j\}) \in \{\text{libre}, \text{surcoût}, \text{interdit}\}$$

Le coût effectif de traversée est :

$$c(\{v_i, v_j\}) = \begin{cases} +\infty & \text{si } \text{statut}(\{v_i, v_j\}) = \text{interdit} \\ w_0(\{v_i, v_j\}) \cdot (1 + \tau_{ij}) & \text{si } \text{statut}(\{v_i, v_j\}) = \text{surcoût}, \; \tau_{ij} > 0 \\ w_0(\{v_i, v_j\}) & \text{sinon} \end{cases}$$

où $\tau_{ij} \in [0, 1]$ est le taux de surcoût de l'arête (e.g., $\tau = 0.3$ pour une route à péage majorant le coût de 30 %).

Une arête interdite est équivalente à une absence d'arête dans le graphe résiduel utilisé pour la résolution. Le graphe résiduel (c'est-à-dire $G$ privé des arêtes interdites) doit rester connexe pour garantir l'existence d'une solution réalisable. Cette propriété sera vérifiée lors de la génération des instances.

#### Contrainte 2 — Capacité de chargement

Un véhicule unique de capacité maximale $Q \in \mathbb{R}_+^*$ effectue la totalité des livraisons. Lorsque sa charge résiduelle ne lui permet plus de satisfaire le prochain client de sa tournée, il retourne au dépôt $v_0$ pour se recharger, puis repart. L'ensemble des livraisons est donc découpé en sous-tournées $R = (r_1, r_2, \dots, r_m)$, où chaque sous-tournée $r_s = (v_0, v_{i_1}, \dots, v_{i_p}, v_0)$ est un circuit partant et revenant au dépôt.

Pour toute sous-tournée $r_s$ :

$$\sum_{j=1}^{p} \delta_{j} \leq Q$$

Le nombre $m$ de sous-tournées n'est pas fixé a priori : il dépend de la répartition des clients et de leurs demandes. Dans le cas uniforme étudié ici, on note

$$m_{\min} =  \frac{(n-1)\delta}{Q} $$

où $m_{\min}$ désigne le nombre minimal théorique de sous-tournées nécessaires. On en déduit alors la borne simple :

$$m \geq m_{\min}$$

#### Contrainte 3 — Pondération dynamique (coûts temporels)

Le coût de traversée d'une arête varie selon l'heure de passage. On discrétise le temps en $|T|$ intervalles horaires :

$$T = \{t_0, t_1, \dots, t_{|T|-1}\}$$

Le coût réel au moment $t$ de franchissement de l'arête $\{v_i, v_j\}$ dépend d'un coefficient dynamique propre à cette arête et à cet instant. Ce coefficient peut augmenter le coût, le diminuer, ou rendre l'arête indisponible sur un intervalle donné :

$$c(\{v_i, v_j\}, t) = \begin{cases} +\infty & \text{si l'arête est indisponible à l'instant } t \\ w_0(\{v_i, v_j\}) \cdot (1 + \tau_{ij}) \cdot \lambda_{ij}(t) & \text{sinon} \end{cases}$$

où $w_0(\{v_i, v_j\})$ est le coût de base, $\tau_{ij}$ le surcoût statique éventuel, et $\lambda_{ij}(t) \in \mathbb{R}_+^*$ le coefficient dynamique appliqué à l'instant $t$. Le cas $0 < \lambda_{ij}(t) < 1$ modélise une période plus favorable que le régime statique.

---

### 2.4 Variables de décision

Pour formuler le problème, on introduit les variables suivantes :

**Variable de routage :**

$$y_{ij} = \begin{cases} 1 & \text{si le véhicule emprunte l'arête } \{v_i, v_j\} \text{ dans sa tournée} \\ 0 & \text{sinon} \end{cases}$$

Cette variable binaire encode la structure de la tournée : quelles arêtes sont empruntées. Puisque le véhicule effectue potentiellement plusieurs sous-tournées (retours au dépôt), une arête peut être empruntée plusieurs fois. On généralise donc : $y_{ij} \in \mathbb{N}$ représente le nombre de fois que l'arête $\{v_i, v_j\}$ est empruntée au total sur l'ensemble des sous-tournées.

**Variable d'affectation :**

$$x_{is} = \begin{cases} 1 & \text{si le client } v_i \text{ est livré lors de la sous-tournée } r_s \\ 0 & \text{sinon} \end{cases}$$

Cette variable encode la répartition des clients entre les sous-tournées. Chaque client doit être affecté à exactement une sous-tournée : $\sum_{s=1}^{m} x_{is} = 1$ pour tout client $v_i$.

**Variable temporelle :**

$$t_{ij} \in T$$

Représente l'instant de passage du véhicule sur l'arête $\{v_i, v_j\}$, nécessaire pour déterminer le coefficient dynamique $\lambda_{ij}(t_{ij})$ applicable.

---

### 2.5 Fonction objectif

On cherche à minimiser la somme des coûts de toutes les sous-tournées, en tenant compte des surcoûts statiques et des variations temporelles :

$$\min \sum_{\{v_i, v_j\} \in E} y_{ij} \cdot c\bigl(\{v_i, v_j\}, t_{ij}\bigr)$$

où :
- $y_{ij}$ est le nombre de passages sur l'arête $\{v_i, v_j\}$
- $c(\{v_i, v_j\}, t_{ij})$ est le coût effectif de traversée au temps $t_{ij}$, intégrant les surcoûts statiques ($\tau_{ij}$) et le coefficient dynamique $\lambda_{ij}(t_{ij})$

---

### 2.6 Formulation synthétique du problème

$$\min \sum_{\{v_i, v_j\} \in E} y_{ij} \cdot c\bigl(\{v_i, v_j\}, t_{ij}\bigr)$$

sous les contraintes :

$$\text{(C1) Couverture :} \quad \sum_{s=1}^{m} x_{is} = 1 \quad \forall\, v_i \in V \setminus \{v_0\}$$

$$\text{(C2) Capacité :} \quad \sum_{v_i \in r_s} \delta_i \leq Q \quad \forall\, s \in \{1, \dots, m\}$$

$$\text{(C3) Restrictions :} \quad y_{ij} = 0 \quad \forall\, \{v_i, v_j\} \text{ tel que } \text{statut}(\{v_i, v_j\}) = \text{interdit}$$

$$\text{(C4) Coûts dynamiques :} \quad c(\{v_i, v_j\}, t_{ij}) = \begin{cases} +\infty & \text{si l'arête n'est pas disponible à } t_{ij} \\ w_0(\{v_i, v_j\}) \cdot (1 + \tau_{ij}) \cdot \lambda_{ij}(t_{ij}) & \text{sinon} \end{cases}$$

$$\text{(C5) Structure :} \quad \text{Chaque sous-tournée } r_s \text{ est un circuit passant par } v_0$$

Ce problème est un problème de tournées de véhicule avec contraintes de capacité, restrictions sur les arêtes et routage dynamique. On le note VRP-CDR (*Vehicle Routing Problem with Capacity, edge Constraints and Dynamic Routing*).

---


## 3. Illustration de la modélisation

Avant d'aborder la théorie de la complexité, illustrons les structures de données sur un exemple jouet : 6 clients, 1 dépôt, 1 véhicule de capacité limitée (qui effectue plusieurs sous-tournées).

In [1]:
import math
import random
import numpy as np
from collections import deque

# ─── Sommets ───
DEPOT = 0
sommets = list(range(7))
n = len(sommets)

# ─── Seed global pour la reproductibilité ───
random.seed(42)

# ─── Demandes ───
DEMANDE_UNIFORME = random.randint(1, 20)
demandes = {0: 0}
for i in range(1, n):
    demandes[i] = DEMANDE_UNIFORME

print(f"Demande uniforme par client : {DEMANDE_UNIFORME}")

# ─── Capacité ───
CAPACITE = 40

# ─── Matrice de coûts de base (matrice d'adjacence pondérée) ───
# matrice_couts_base[i][j] = coût de la route directe entre i et j.
# 0 hors diagonale = pas de route directe.
# Matrice symétrique (graphe non orienté).
# Le graphe complet exploité ensuite est reconstruit par plus courts chemins.

matrice_couts_base = np.array([
#    v0    v1    v2    v3    v4    v5    v6
    [0.0,  3.6,  4.1,  4.1,  3.6,  3.2,  2.8],   # v0 (dépôt)
    [3.6,  0.0,  3.2,  2.8,  5.1,  5.4,  4.1],   # v1
    [4.1,  3.2,  0.0,  5.8,  7.2,  5.4,  2.2],   # v2
    [4.1,  2.8,  5.8,  0.0,  3.2,  7.3,  6.1],   # v3
    [3.6,  5.1,  7.2,  3.2,  0.0,  6.1,  6.4],   # v4
    [3.2,  5.4,  5.4,  7.3,  6.1,  0.0,  3.2],   # v5
    [2.8,  4.1,  2.2,  6.1,  6.4,  3.2,  0.0],   # v6
])

def cout_base(u, v):
    """Coût de base entre deux sommets (lecture de la matrice d'adjacence)."""
    return matrice_couts_base[u][v]

# ─── Contrainte 1 : restrictions (aléatoires, seed fixée) ───
random.seed(42)
arcs_interdits = set()
surcoûts = {}

for i in range(n):
    for j in range(i + 1, n):
        tirage = random.random()
        if tirage < 0.10:
            arcs_interdits.add((i, j))
            arcs_interdits.add((j, i))
        elif tirage < 0.30:
            taux = round(random.uniform(0.10, 0.50), 2)
            surcoûts[(i, j)] = taux
            surcoûts[(j, i)] = taux

# Vérification connexité
def verifier_connexite(n, arcs_interdits):
    """Vérifie que le graphe résiduel est connexe par BFS depuis le dépôt."""
    adj = {i: set() for i in range(n)}
    for i in range(n):
        for j in range(n):
            if i != j and (i, j) not in arcs_interdits:
                adj[i].add(j)
    visites = set()
    file = deque([0])
    visites.add(0)
    while file:
        u = file.popleft()
        for v in adj[u]:
            if v not in visites:
                visites.add(v)
                file.append(v)
    return len(visites) == n

assert verifier_connexite(n, arcs_interdits), \
    "Le graphe résiduel n'est pas connexe ! Régénérer l'instance."

def cout_effectif(u, v):
    """
    Coût effectif d'une arête (u, v) en tenant compte des restrictions.
    Retourne float('inf') si l'arête est interdite.
    """
    if (u, v) in arcs_interdits:
        return float('inf')
    base = cout_base(u, v)
    taux = surcoûts.get((u, v), 0.0)
    return round(base * (1 + taux), 2)

# ─── Contrainte 3 : pondération dynamique ───
TRANCHES_HORAIRES = {
    "matin"        : (6,  8,  0.95),
    "pointe_matin" : (8,  10, 1.30),
    "journee"      : (10, 16, 0.90),
    "pointe_soir"  : (16, 19, 1.40),
    "soir"         : (19, 22, 0.85),
}

def coefficient_dynamique(heure):
    """Retourne le coefficient dynamique λ(t) pour une heure donnée."""
    for nom, (h_debut, h_fin, lam) in TRANCHES_HORAIRES.items():
        if h_debut <= heure < h_fin:
            return lam
    return 1.0

def cout_dynamique(u, v, heure):
    """Coût réel intégrant restrictions statiques et variations temporelles."""
    c_eff = cout_effectif(u, v)
    if c_eff == float('inf'):
        return float('inf')
    return round(c_eff * coefficient_dynamique(heure), 2)

# ─── Affichage ───
print("=== Paramètres du problème ===")
print(f"Sommets : {sommets}  (0 = dépôt)")
print(f"Capacité du véhicule : {CAPACITE}")
demande_totale = sum(demandes.values())
nb_sous_tournees_min = math.ceil(demande_totale / CAPACITE)
print(f"Demande totale : {demande_totale}")
print(f"Nombre minimal de sous-tournées : {nb_sous_tournees_min}")
print(f"\nArêtes interdites : {arcs_interdits}")
print(f"Arêtes avec surcoût : {surcoûts}")
print()

# Exemples de coûts dynamiques
print("=== Exemples de coûts ===")
exemple_interdit = exemple_surtaxe = exemple_libre = None
for i in range(n):
    for j in range(i + 1, n):
        if (i, j) in arcs_interdits and not exemple_interdit:
            exemple_interdit = (i, j)
        elif (i, j) in surcoûts and not exemple_surtaxe:
            exemple_surtaxe = (i, j)
        elif not exemple_libre:
            exemple_libre = (i, j)

if exemple_libre:
    u, v = exemple_libre
    print(f"Arête libre ({u}→{v}) à 9h : base={cout_base(u,v)}, coût={cout_dynamique(u,v,9)}")
if exemple_surtaxe:
    u, v = exemple_surtaxe
    print(f"Arête avec surcoût ({u}→{v}) à 14h : base={cout_base(u,v)}, τ={surcoûts[(u,v)]}, coût={cout_dynamique(u,v,14)}")
if exemple_interdit:
    u, v = exemple_interdit
    print(f"Arête interdite ({u}→{v}) à 11h : coût={cout_dynamique(u,v,11)}")
if exemple_libre:
    u, v = exemple_libre
    print(f"Arête libre ({u}→{v}) à 17h : coût={cout_dynamique(u,v,17)}")

Demande uniforme par client : 4
=== Paramètres du problème ===
Sommets : [0, 1, 2, 3, 4, 5, 6]  (0 = dépôt)
Capacité du véhicule : 40
Demande totale : 24
Nombre minimal de sous-tournées : 1

Arêtes interdites : {(1, 2), (2, 1), (3, 4), (4, 3), (6, 1), (2, 0), (1, 4), (0, 2), (1, 6), (4, 1)}
Arêtes avec surcoût : {(0, 3): 0.19, (3, 0): 0.19, (1, 5): 0.3, (5, 1): 0.3, (2, 3): 0.36, (3, 2): 0.36, (2, 5): 0.34, (5, 2): 0.34, (4, 6): 0.48, (6, 4): 0.48}

=== Exemples de coûts ===
Arête libre (0→1) à 9h : base=3.6, coût=4.68
Arête avec surcoût (0→3) à 14h : base=4.1, τ=0.19, coût=4.39
Arête interdite (0→2) à 11h : coût=inf
Arête libre (0→1) à 17h : coût=5.04


In [7]:
import heapq

# ─── Construction du graphe complet via Dijkstra ───
# Un appel depuis une source : O(n² log n) dans cette implémentation.
# Construction de toute la matrice complète : O(n³ log n).

def dijkstra(source, n, matrice_couts_base, arcs_interdits):
    """Plus courts chemins depuis source vers tous les sommets."""
    dist = [float('inf')] * n
    dist[source] = 0
    pq = [(0, source)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue
        for v in range(n):
            if v == u or (u, v) in arcs_interdits:
                continue
            cout = matrice_couts_base[u][v]
            if cout == 0:
                continue
            nd = d + cout
            if nd < dist[v]:
                dist[v] = nd
                heapq.heappush(pq, (nd, v))
    return dist

matrice_complete = np.zeros((n, n))
for i in range(n):
    distances_depuis_i = dijkstra(i, n, matrice_couts_base, arcs_interdits)
    for j in range(n):
        matrice_complete[i][j] = round(distances_depuis_i[j], 2)

print("=== Matrice des plus courts chemins (graphe complet) ===")
print(matrice_complete)

=== Matrice des plus courts chemins (graphe complet) ===
[[0.  3.6 5.  4.1 3.6 3.2 2.8]
 [3.6 0.  8.6 2.8 7.2 5.4 6.4]
 [5.  8.6 0.  5.8 7.2 5.4 2.2]
 [4.1 2.8 5.8 0.  7.7 7.3 6.1]
 [3.6 7.2 7.2 7.7 0.  6.1 6.4]
 [3.2 5.4 5.4 7.3 6.1 0.  3.2]
 [2.8 6.4 2.2 6.1 6.4 3.2 0. ]]


## 4. Analyse de complexité

### 4.1 Rappels sur les classes de complexité

Avant de démontrer la complexité de notre problème, rappelons les définitions fondamentales :

| Classe | Définition |
|--------|------------|
| **P** | Problèmes résolubles en temps *polynomial* par un algorithme déterministe |
| **NP** | Problèmes dont une solution peut être *vérifiée* en temps polynomial |
| **NP-Difficile** | Tout problème de NP se réduit polynomialement à ce problème |
| **NP-Complet** | Problème à la fois dans NP et NP-Difficile |

La question ouverte $P \stackrel{?}{=} NP$ reste l'un des plus grands problèmes non résolus en informatique théorique. La quasi-totalité des experts considère que $P \neq NP$, ce qui signifie que les problèmes NP-Complets n'admettent **aucun algorithme polynomial**.

---

### 4.2 Problème de décision associé

Pour analyser la complexité, on reformule d'abord notre problème d'optimisation en problème de décision (réponse binaire oui/non) :

**VRP-CDR-DEC** : Étant donné un graphe $G = (V, E, w_0)$, un véhicule de capacité $Q$, des demandes $\delta_i$, des surcoûts statiques $\tau_{ij}$, des coefficients dynamiques $\lambda_{ij}(t)$ et une borne de coût $C^* \in \mathbb{R}_+^*$, existe-t-il un ensemble de sous-tournées réalisables couvrant tous les clients avec un coût total $\leq C^*$ ?

---

### 4.3 Le VRP-CDR est dans NP

**Théorème :** VRP-CDR-DEC $\in$ NP.

**Preuve :** Un certificat est donné par la liste ordonnée des sous-tournées $R = (r_1, r_2, \dots, r_m)$, ainsi que par les instants de passage associés aux arêtes empruntées.

Sa taille est polynomiale en $n$ : chaque client apparaît une seule fois, chaque sous-tournée commence et se termine au dépôt, et le nombre total d'occurrences de sommets dans le certificat reste donc linéaire en fonction du nombre de clients.

La vérification se fait alors en temps polynomial :

1. Vérifier que chaque client apparaît exactement une fois et que tous les clients sont couverts → $O(n)$
   Cette vérification consiste à parcourir une fois la liste des clients présents dans le certificat.
2. Pour chaque sous-tournée $r_s$, additionner les demandes des clients qu'elle contient et vérifier que cette somme ne dépasse pas $Q$ → $O(n)$
   Là encore, chaque client est lu au plus une fois lorsqu'on additionne les demandes de sa sous-tournée.
3. Pour chaque arête empruntée, vérifier qu'elle est autorisée au temps considéré et calculer son coût effectif par lecture de $w_0$, $\tau_{ij}$ et $\lambda_{ij}(t)$ → $O(n)$
   Le nombre d'arêtes parcourues dans le certificat est du même ordre que le nombre total de visites, et chaque lecture de coût se fait en temps constant.
4. Additionner les coûts de toutes les sous-tournées et comparer le total à $C^*$ → $O(n)$
   On additionne simplement une fois les coûts déjà calculés le long du certificat.

Au total, la vérification est polynomiale, et même linéaire en la taille du certificat lorsque les valeurs dynamiques sont accessibles en temps constant. Donc **VRP-CDR-DEC $\in$ NP**. CQFD

---

### 4.4 Le VRP-CDR est NP-Difficile : réduction depuis le Δ-TSP

**Stratégie :** Nous allons montrer que le **Δ-TSP** (Problème du Voyageur de Commerce métrique, connu NP-Complet) se réduit polynomialement au VRP-CDR. Cela prouvera que VRP-CDR est au moins aussi difficile que le Δ-TSP.

**Rappel du Δ-TSP :** Étant donné un graphe complet pondéré $G' = (V, E', w')$ respectant l'inégalité triangulaire ($w'(u,v) \leq w'(u,z) + w'(z,v)$ pour tout $z$), existe-t-il un cycle hamiltonien de coût $\leq C^*$ ?

Le Δ-TSP est NP-Complet d'après *Garey et Johnson*<sup>3</sup>.

#### Construction de la réduction

Soit une instance du Δ-TSP : graphe complet $G' = (V, E', w')$ avec $|V| = n$, borne $B$.

On construit une instance du VRP-CDR de la façon suivante :

1. **Dépôt et clients :** on choisit un sommet $v_0$ comme dépôt et les $n-1$ autres sommets deviennent les clients
2. **Demandes :** $\delta_i = 1$ pour tout client $v_i$
3. **Capacité :** $Q = n - 1$, de sorte qu'un seul chargement permet de desservir tous les clients
4. **Restrictions et dynamique :** toutes les arêtes sont autorisées, $\tau_{ij} = 0$ et $\lambda_{ij}(t) = 1$ pour toute arête et tout instant
5. **Coûts :** le coût de chaque arête est conservé, c'est-à-dire $c(\{v_i, v_j\}, t) = w'(\{v_i, v_j\})$
6. **Borne :** $C^* = B$

Cette transformation est réalisée en **$O(n^2)$** : l'instance de départ est un graphe complet, donc elle contient $\Theta(n^2)$ coûts d'arêtes à recopier. Les autres opérations consistent seulement à affecter $n-1$ demandes unitaires et à fixer un nombre constant de paramètres. Le terme dominant est donc la copie de la matrice des coûts, ce qui justifie la borne $O(n^2)$.

#### Preuve de correction

Dans l'instance construite, la somme des demandes des clients vaut $n - 1$ et la capacité du véhicule vaut également $Q = n - 1$. Le véhicule peut donc desservir tous les clients en une seule sous-tournée. De plus, tous les coûts sont strictement positifs ; par conséquent, un retour intermédiaire au dépôt ajouterait nécessairement un coût positif supplémentaire. Une solution optimale de l'instance construite n'a donc aucun intérêt à se décomposer en plusieurs sous-tournées.

Toute solution optimale du VRP-CDR construit est ainsi une unique sous-tournée de la forme $$(v_0, v_{\sigma(1)}, \dots, v_{\sigma(n-1)}, v_0),$$ où chaque client apparaît exactement une fois. Une telle sous-tournée est précisément un cycle hamiltonien dans le graphe complet $G'$.

Réciproquement, tout cycle hamiltonien de $G'$ définit immédiatement une sous-tournée réalisable du VRP-CDR construit, puisque les demandes sont unitaires, que la capacité autorise la visite de tous les clients en un seul trajet, et qu'aucune arête n'est interdite.

Enfin, les coûts sont inchangés par la transformation : on a $c(\{v_i, v_j\}, t) = w'(\{v_i, v_j\})$ pour toute arête. Le coût total de la sous-tournée dans l'instance VRP-CDR est donc exactement égal au coût du cycle correspondant dans l'instance de Δ-TSP.

On obtient donc l'équivalence suivante : l'instance de Δ-TSP admet un cycle hamiltonien de coût inférieur ou égal à $B$ si et seulement si l'instance construite de VRP-CDR-DEC admet un ensemble de sous-tournées réalisables de coût total inférieur ou égal à $C^*$. La réduction est correcte, donc $\Delta\text{-TSP} \leq_p \text{VRP-CDR}$ et **VRP-CDR est NP-Difficile**. CQFD

---

### 4.5 Conclusion : NP-Complétude du VRP-CDR

Nous avons montré que **VRP-CDR-DEC appartient à NP** car un certificat, constitué d'un ensemble de sous-tournées et des instants de passage associés, peut être vérifié en temps polynomial. Nous avons également établi que **VRP-CDR-DEC est NP-Difficile** par réduction polynomiale depuis le Δ-TSP<sup>3</sup>. Par conséquent, **VRP-CDR-DEC est NP-Complet**.

Le problème d'optimisation associé, qui consiste à minimiser la somme des coûts des sous-tournées réalisables, est donc au moins aussi difficile que sa version décisionnelle. Ce résultat est cohérent avec la littérature classique sur les problèmes de tournées de véhicules, introduits notamment par Dantzig et Ramser<sup>4</sup>.


In [9]:
import itertools

# ─────────────────────────────────────────────────────────────
# Démonstration de la réduction Δ-TSP → VRP-CDR
# ─────────────────────────────────────────────────────────────

def reduction_tsp_vers_vrp(sommets_tsp, distances_tsp):
    """
    Construit une instance VRP-CDR équivalente à une instance Δ-TSP.
    
    Le véhicule unique a une capacité = n-1 (peut tout livrer),
    les demandes sont unitaires, sans restriction ni coefficient dynamique supplémentaire.
    
    Retourne (instance_vrp, borne_B) où instance_vrp est un dict
    décrivant le problème VRP-CDR correspondant.
    """
    n = len(sommets_tsp)
    depot = sommets_tsp[0]
    clients = sommets_tsp[1:]

    instance_vrp = {
        "depot"         : depot,
        "clients"       : clients,
        "demandes"      : {v: 1 for v in clients},   # charges unitaires
        "capacite"      : n - 1,                     # un seul véhicule suffit
        "distances"     : distances_tsp,             # coûts identiques au TSP
        "arcs_interdits": set(),                     # aucune restriction
        "surtaxes"      : {},                        # aucun surcoût
        "coefficient_dynamique": lambda t: 1.0,      # dynamique neutre
    }
    return instance_vrp


def cycle_hamiltonien_cout(sommets, distances):
    """
    Calcule le coût du meilleur cycle hamiltonien par force brute.
    Utilisé uniquement pour valider la réduction sur de petits exemples (n ≤ 8).
    Complexité : O((n-1)!) — non utilisable en production.
    """
    depot = sommets[0]
    clients = sommets[1:]
    meilleur = float('inf')
    meilleure_perm = None

    for perm in itertools.permutations(clients):
        tournee = [depot] + list(perm) + [depot]
        cout = sum(
            distances.get(
                (min(tournee[i], tournee[i+1]), max(tournee[i], tournee[i+1])), float('inf')
            )
            for i in range(len(tournee) - 1)
        )
        if cout < meilleur:
            meilleur = cout
            meilleure_perm = tournee

    return meilleur, meilleure_perm


# ── Exemple de validation ──────────────────────────────────────
# Graphe Δ-TSP à 5 sommets (graphe complet, inégalité triangulaire respectée)
sommets_tsp = [0, 1, 2, 3, 4]
distances_tsp = {
    (0, 1): 10, (0, 2): 15, (0, 3): 20, (0, 4): 25,
    (1, 2):  8, (1, 3): 12, (1, 4): 18,
    (2, 3):  9, (2, 4): 14,
    (3, 4):  7,
}

print("=== Validation de la réduction Δ-TSP → VRP-CDR ===")
print(f"Sommets TSP : {sommets_tsp}")
print()

# Résolution du TSP
cout_tsp, cycle_tsp = cycle_hamiltonien_cout(sommets_tsp, distances_tsp)
print(f"[Δ-TSP]   Meilleur cycle hamiltonien : {cycle_tsp}")
print(f"[Δ-TSP]   Coût optimal : {cout_tsp}")
print()

# Construction de l'instance VRP-CDR équivalente
instance_vrp = reduction_tsp_vers_vrp(sommets_tsp, distances_tsp)
print("[VRP-CDR] Instance construite :")
print(f"  Dépôt       : {instance_vrp['depot']}")
print(f"  Clients     : {instance_vrp['clients']}")
print(f"  Demandes    : {instance_vrp['demandes']}")
print(f"  Capacité    : {instance_vrp['capacite']}")
print()

# Validation : la tournée optimale du VRP-CDR correspond au cycle hamiltonien du TSP
cout_vrp, tournee_vrp = cycle_hamiltonien_cout(sommets_tsp, instance_vrp["distances"])
print(f"[VRP-CDR] Tournée optimale unique : {tournee_vrp}")
print(f"[VRP-CDR] Coût : {cout_vrp}")
print()
print(f"Équivalence validée : {cout_tsp == cout_vrp} (coûts identiques)")
print("=> La réduction est correcte : Δ-TSP ≤p VRP-CDR")

=== Validation de la réduction Δ-TSP → VRP-CDR ===
Sommets TSP : [0, 1, 2, 3, 4]

[Δ-TSP]   Meilleur cycle hamiltonien : [0, 1, 3, 4, 2, 0]
[Δ-TSP]   Coût optimal : 58

[VRP-CDR] Instance construite :
  Dépôt       : 0
  Clients     : [1, 2, 3, 4]
  Demandes    : {1: 1, 2: 1, 3: 1, 4: 1}
  Capacité    : 4

[VRP-CDR] Tournée optimale unique : [0, 1, 3, 4, 2, 0]
[VRP-CDR] Coût : 58

Équivalence validée : True (coûts identiques)
=> La réduction est correcte : Δ-TSP ≤p VRP-CDR


## 5. Conséquences pratiques et choix algorithmiques

La NP-Complétude du VRP-CDR implique (sous l'hypothèse $P \neq NP$) :

- **Aucun algorithme polynomial** ne peut garantir de trouver la solution optimale
- Tout algorithme **exact** a une complexité au moins exponentielle dans le pire cas
- Pour $n$ clients, l'espace des solutions est de l'ordre de $n!$ — absolument impraticable dès $n \geq 20$

En force-brute, la réalisation prendrait trop de temps, il faut alors partir sur un algorithme heuristique.

---

## 6. Conclusion

Ce premier livrable a permis de formaliser le problème étudié comme un VRP-CDR sur un graphe non orienté pondéré, avec un véhicule unique, des contraintes de capacité, des restrictions sur les arêtes et une pondération dynamique des coûts. La modélisation met en évidence que l'objectif n'est pas seulement de trouver une tournée réalisable, mais bien de minimiser la somme des coûts de l'ensemble des sous-tournées nécessaires pour desservir tous les clients.

L'analyse de complexité montre que la version décisionnelle du problème est NP-Complète. En pratique, cela signifie qu'il ne faut pas espérer une méthode exacte efficace pour des instances de grande taille. Cette conclusion justifie le choix, dans les livrables suivants, de méthodes heuristiques ou métaheuristiques capables de produire rapidement de bonnes solutions.

Le présent document constitue donc une base théorique solide pour la suite du projet : les hypothèses sont explicitées, les notations sont fixées, les contraintes sont formalisées et le niveau de difficulté algorithmique du problème est établi.


## 7. Glossaire des notations

| Symbole | Type | Description |
|---------|------|-------------|
| $G = (V, E, w_0)$ | Graphe | Graphe non orienté pondéré du réseau routier |
| $G' = (V, E', w')$ | Graphe | Graphe complet obtenu après complétion par plus courts chemins |
| $V$ | Ensemble | Sommets : dépôt $v_0$ et clients $v_1, \dots, v_{n-1}$ |
| $E$ | Ensemble | Arêtes : routes existantes entre les sommets |
| $E'$ | Ensemble | Arêtes du graphe complet |
| $n$ | $\mathbb{N}^*$ | Nombre total de sommets ($|V|$) |
| $w_0(\{v_i, v_j\})$ | $\mathbb{R}_+^*$ | Coût de base de l'arête $\{v_i, v_j\}$ |
| $w'(\{v_i, v_j\})$ | $\mathbb{R}_+^*$ | Coût du plus court chemin entre $v_i$ et $v_j$ |
| $\delta$ | $\mathbb{R}_+$ | Demande uniforme d'un client |
| $\delta_i$ | $\mathbb{R}_+$ | Demande du client $v_i$ |
| $Q$ | $\mathbb{R}_+^*$ | Capacité maximale du véhicule |
| $\tau_{ij}$ | $[0,1]$ | Taux de surcoût statique de l'arête $\{v_i, v_j\}$ |
| $\lambda_{ij}(t)$ | $\mathbb{R}_+^*$ | Coefficient dynamique appliqué à l'arête $\{v_i, v_j\}$ à l'instant $t$ |
| $T$ | Ensemble | Ensemble des intervalles temporels |
| $c(\{v_i, v_j\}, t)$ | $\mathbb{R}_+^* \cup \{+\infty\}$ | Coût effectif de traversée |
| $y_{ij}$ | $\mathbb{N}$ | Nombre de passages sur l'arête $\{v_i, v_j\}$ |
| $x_{is}$ | $\{0, 1\}$ | 1 si le client $v_i$ est livré dans la sous-tournée $r_s$ |
| $t_{ij}$ | $T$ | Instant de passage sur l'arête $\{v_i, v_j\}$ |
| $R = (r_1, \dots, r_m)$ | Séquence | Ensemble des sous-tournées |
| $m$ | $\mathbb{N}^*$ | Nombre de sous-tournées |
| $C^*$ | $\mathbb{R}_+^*$ | Borne de coût pour le problème de décision |


## 8. Références bibliographiques

1. **Dijkstra, E. W.** (1959). *A note on two problems in connexion with graphs*. *Numerische Mathematik*, 1, 269-271.

2. **Fredman, M. L., & Tarjan, R. E.** (1987). *Fibonacci heaps and their uses in improved network optimization algorithms*. *Journal of the ACM*, 34(3), 596-615.

3. **Garey, M. R., & Johnson, D. S.** (1979). *Computers and Intractability: A Guide to the Theory of NP-Completeness*. W.H. Freeman and Company.

4. **Dantzig, G. B., & Ramser, J. H.** (1959). *The Truck Dispatching Problem*. *Management Science*, 6(1), 80-91.

5. **Toth, P., & Vigo, D.** (Eds.) (2014). *Vehicle Routing: Problems, Methods, and Applications* (2nd ed.). SIAM.

6. **Christofides, N.** (1976). *Worst-case analysis of a new heuristic for the travelling salesman problem*. Technical Report, Carnegie Mellon University.

7. **Cordeau, J.-F., Laporte, G., Savelsbergh, M. W. P., & Vigo, D.** (2007). *Vehicle Routing*. In *Handbooks in Operations Research and Management Science*, Vol. 14, pp. 367-428. Elsevier.
